***<H1>LSTM Based</H1>***

***Subject dependent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_trial(seed_value = 1737200047):

    set_seed(seed_value)
   
    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels and difficulty bins

    # Define the LSTM Model
    class LSTMModel(nn.Module):
        def __init__(self, input_size, hidden_size, output_size, num_layers=1):
            super(LSTMModel, self).__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(hidden_size*2, output_size)

        def forward(self, x):
            # LSTM layer
            lstm_out, _ = self.lstm(x)
            # Use the output of the last LSTM time step
            lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
            # Fully connected layer
            out = self.fc(lstm_out_last)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts):
        """
        Train the LSTM model using random curriculum learning where data is shuffled and split into 25% increments.
        """
        hidden_size = 128
        num_layers = 2
        batch_size = 8
        max_learning_rate = 5e-4
        min_learning_rate = 5e-5
        epochs = epoch_counts

        # Initialize the full dataset
        full_dataset = AudioDataset(audio_folder)
        input_size = full_dataset.input_size
        output_size = len(full_dataset.label_map)
        scaler = full_dataset.scaler
        
        # Shuffle the dataset indices to create random curriculum
        dataset_size = len(full_dataset)
        indices = list(range(dataset_size))
        random.shuffle(indices)
        
        # Split into 25% increments
        split_points = [int(dataset_size * 0.25), int(dataset_size * 0.5), int(dataset_size * 0.75)]
        stage_indices = [
            indices[:split_points[0]],  # First 25% - "Easy"
            indices[:split_points[1]],  # First 50% - "Easy" + "Borderline Easy"
            indices[:split_points[2]],  # First 75% - "Easy" + "Borderline Easy" + "Borderline Tough"
            indices  # 100% - All data ("Tough" added)
        ]
        
        stage_names = [
            "Random Easy (25%)",
            "Random Easy + Borderline Easy (50%)",
            "Random Easy + Borderline Easy + Borderline Tough (75%)",
            "Random Easy + Borderline Easy + Borderline Tough + Tough (100%)"
        ]
        
        print(f"Number of total Training data: {dataset_size}")
        print(f"Data splits: {[len(indices) for indices in stage_indices]}")

        # Initialize the model and optimizer
        model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

        # Training across stages
        for stage, (indices, stage_name) in enumerate(zip(stage_indices, stage_names), 1):
            print(f"\nTraining Stage {stage} with {stage_name} for random curriculum")

            for param_group in optimizer.param_groups:
                param_group['lr'] = max_learning_rate

            # Initialize scheduler for the current stage
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 
                                                                T_max=epoch_counts[stage - 1], 
                                                                eta_min=min_learning_rate)
            
            # Create subset and dataloader for the current stage
            subset = Subset(full_dataset, indices)
            print(f"Number of Training data for stage {stage}: {len(subset)}")

            train_loader = DataLoader(subset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

            # Training loop for current stage
            loss_values = []
            for epoch in range(epochs[stage - 1]):
                model.train()
                running_loss = 0.0
                for i, (inputs, labels) in enumerate(train_loader):  # Ignore difficulty bins in random curriculum
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Forward pass
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimization
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item()

                # Update LR scheduler
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']

                epoch_loss = running_loss / len(train_loader)
                loss_values.append(epoch_loss)
                print(f"Stage {stage}, Epoch {epoch + 1}/{epochs[stage - 1]}, Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}")

            # Save model, loss, and scaler for the current stage
            save_dir = f"saved_weights/subject_dependent/{scenario_name}/lstm/random_seed_{seed_value}/stage_{stage}"
            os.makedirs(save_dir, exist_ok=True)
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': scaler
            }
            torch.save(checkpoint, f"{save_dir}/model_checkpoint.pth")
            print(f"Checkpoint saved for random curriculum, Stage {stage}")

    # Define training scenarios
    scenarios = {
        "random_curriculum": [50, 50, 50, 50] 
    }

    # Start training
    audio_folder = 'subject_ordered_features/Training'
    for scenario_name, epoch_counts in scenarios.items():
        print(f"\nStarting training for {scenario_name} with epoch counts: {epoch_counts}")
        random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts)

random_seed = 1737200047

print(f"Starting trial with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Dependent - Testing***

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
from torch.utils.data import Dataset
import seaborn as sns

random_seed = 1737200047
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size*2, output_size)

    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        # Use the output of the last LSTM time step
        lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
        # Fully connected layer
        out = self.fc(lstm_out_last)
        return out

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=None, modality="audio"):
        """
        Args:
        - audio_folder: Path to the testing data folder.
        - curriculum_csv_file: CSV file with difficulty scores and bins.
        - scaler: Fitted scaler from the training data.
        - difficulty_score: The difficulty score being evaluated (e.g., "true_label_score").
        - bin_filter: List of bins to filter testing data by (e.g., ["Easy"]).
        - modality: Modality of the testing data (e.g., "audio").
        """
        self.audio_folder = audio_folder
        self.csv_file = curriculum_csv_file
        self.scaler = scaler
        self.difficulty_score = difficulty_score
        self.bin_filter = bin_filter
        self.modality = modality

        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()

        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        # Read and filter CSV data
        df = pd.read_csv(self.csv_file)
        df = df[df['modality'] == self.modality]  # Filter by modality

        # Map difficulty scores to their corresponding bin columns
        bin_column_mapping = {
            'intended_emotion_score': 'bin_intended_emotion_score',
            'entropy_score': 'bin_entropy_score',
            'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
            'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
            'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
        }
        bin_column = bin_column_mapping.get(self.difficulty_score)

        if self.bin_filter:
            df = df[df[bin_column].isin(self.bin_filter)]  # Filter by the selected bins

        file_paths = []
        labels = []

        for _, row in df.iterrows():
            subject_id = row['fileName'].split('_')[0]
            filename = os.path.basename(row['fileName'])
            full_path = os.path.join(self.audio_folder, subject_id, 'Audio', f"{filename}.csv")

            if os.path.exists(full_path):
                full_path = os.path.join('hubert_features', f"{filename}.npy")
                file_paths.append(full_path)
                labels.append(self.extract_label_from_filename(filename))

        return file_paths, labels

    def extract_label_from_filename(self, filename):
        return filename.split('_')[2]

    def load_csv_data(self, file_path):
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)

    def get_counts(self):
        return len(self.file_paths)

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []
    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

def print_mean_macro_accuracies(mean_macro_accuracies):
    print("\n" + "="*80)
    print("MEAN MACRO ACCURACIES ACROSS TRIALS")
    print("="*80)
    
    for difficulty_score, stages_data in mean_macro_accuracies.items():
        print(f"\nDifficulty Score: {difficulty_score}")
        print("-"*60)
        
        # Print overall macro accuracy
        print("\nOverall Performance:")
        for stage, acc_data in stages_data['overall'].items():
            mean_acc = acc_data['mean']
            std_dev = acc_data['std']
            print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")
        
        # Print bin-wise macro accuracy
        bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
        for bin_name in bins:
            print(f"\nBin: {bin_name}")
            for stage, acc_data in stages_data['bins'][bin_name].items():
                mean_acc = acc_data['mean']
                std_dev = acc_data['std']
                print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")

# Main testing logic
def test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name='intended_emotion_lstm'):
    """
    Test the model for all difficulty scores at each stage of training across bins and on the overall test set.
    Args:
    - checkpoint_base_dir: Base directory containing model checkpoints for each difficulty score.
    - test_folder: Path to the folder containing test data.
    - curriculum_csv_file: Path to the CSV file with difficulty scores and bins.
    - device: Device to run the evaluation on (CPU/GPU).
    - difficulty_scores: List of difficulty scores to evaluate.
    """
    
    # Initialize dictionary to store macro accuracies across trials
    all_trials_macro_acc = {
        score: {
            'overall': {stage: [] for stage in range(1, 5)},
            'bins': {
                bin_name: {stage: [] for stage in range(1, 5)} 
                for bin_name in ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
            }
        } for score in difficulty_scores
    }
    
    print(f'\n{"="*60}')
    print(f'Results for Random Seed: {random_seed}')
    print(f'{"="*60}')

    bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
    results = {}

    for difficulty_score in difficulty_scores:
        print(f"\nStarting evaluation for difficulty score: {scenario_name}")
        checkpoint_dir = os.path.join(checkpoint_base_dir, scenario_name)
        results[difficulty_score] = {
            "overall": [],
            "macro_acc": [],
            "bal_acc": [],
            "bins": {bin_name: [] for bin_name in bins},
            "bins_macro": {bin_name: [] for bin_name in bins},
            "counts": {bin_name: 0 for bin_name in bins},
            "total_count": 0
        }

        loss_curves = []

        for stage in range(1, 5):
            print(f"\nEvaluating model after Stage {stage} training for {scenario_name}...")
                
            checkpoint_path = os.path.join(checkpoint_dir, f"lstm/random_seed_{random_seed}/stage_{stage}/model_checkpoint.pth")
            checkpoint = torch.load(checkpoint_path, weights_only=False)

            # Load model
            input_size = checkpoint['model_state_dict']['lstm.weight_ih_l0'].shape[1]
            hidden_size = 128
            num_layers = 2
            output_size = len(checkpoint['model_state_dict']['fc.bias'])

            model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
            model.load_state_dict(checkpoint['model_state_dict'])
                

            # Load scaler from checkpoint
            scaler = checkpoint['scaler']

            # Overall performance
            overall_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score)
            overall_accuracy, macro_acc, bal_acc, overall_report, overall_labels, overall_preds = evaluate_model(model, overall_dataset, device)
            overall_count = overall_dataset.get_counts()
            results[difficulty_score]["overall"].append(overall_accuracy)
            results[difficulty_score]["macro_acc"].append(macro_acc)
            results[difficulty_score]["bal_acc"].append(bal_acc)
            results[difficulty_score]["total_count"] = overall_count
                
            # Store macro accuracy for this trial and stage
            all_trials_macro_acc[difficulty_score]['overall'][stage].append(macro_acc)
                
            print(f"Overall accuracy after Stage {stage}: {overall_accuracy:.4f}")
            print(f"Macro accuracy after Stage {stage}: {macro_acc:.4f}")
            print(f"Balanced accuracy after Stage {stage}: {bal_acc:.4f}")
            print(f"Total test data points: {overall_count}")

            print("Classification Report:")
            print(overall_report)
                
            # Plot confusion matrix for the final stage
            if stage == 4:
                plot_confusion_matrix(overall_labels, overall_preds, 
                                    overall_dataset.label_map,
                                    title=f'Confusion Matrix {scenario_name} (LSTMs)')

            # Bin-wise performance
            for bin_name in bins:
                bin_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=[bin_name])
                bin_accuracy, macro_acc, _, bin_report, bin_labels, bin_preds = evaluate_model(model, bin_dataset, device)
                bin_count = bin_dataset.get_counts()
                results[difficulty_score]["bins"][bin_name].append(bin_accuracy)
                results[difficulty_score]["bins_macro"][bin_name].append(macro_acc)
                results[difficulty_score]["counts"][bin_name] = bin_count
                    
                # Store bin-wise macro accuracy for this trial and stage
                all_trials_macro_acc[difficulty_score]['bins'][bin_name][stage].append(macro_acc)
                    
                print(f"Std. Accuracy for bin {bin_name} after Stage {stage}: {bin_accuracy:.4f}")
                print(f"Macro Accuracy for bin {bin_name} after Stage {stage}: {macro_acc:.4f}")
                print(f"Number of test data points in bin {bin_name}: {bin_count}")

            # Extract loss values for plotting
            loss_values = checkpoint.get('loss_values', None)
            if loss_values:
                loss_curves.append(loss_values)
            else:
                print(f"Loss values not found in checkpoint for Stage {stage}")

        # Plot loss curves for the difficulty score
        plt.figure(figsize=(10, 6))
        for stage, loss_values in enumerate(loss_curves, start=1):
            plt.plot(range(len(loss_values)), loss_values, label=f"Stage {stage}")
        plt.title(f"Training Loss Curve for {scenario_name}")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["overall"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Performance Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot macro accuracy results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["macro_acc"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins_macro"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Macro Accuracy Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Macro Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

# Paths
scenario_name = "random_curriculum"
checkpoint_base_dir = f"saved_weights/subject_dependent"
test_folder = "subject_ordered_features/Testing"
curriculum_csv_file = "curriculum_dataset_results_binned.csv"
difficulty_scores = {
    'intended_emotion_score': 'bin_intended_emotion_score',
    # 'entropy_score': 'bin_entropy_score',
    # 'intended_emotion_similarity_score': 'bin_intended_emotion_similarity_score',
    # 'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
    # 'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
    # 'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
}

# Run the test
test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name=scenario_name)

***Subject Independent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_trial(seed_value = 1737200047):

    set_seed(seed_value)
   
    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels and difficulty bins

    # Define the LSTM Model
    class LSTMModel(nn.Module):
        def __init__(self, input_size, hidden_size, output_size, num_layers=1):
            super(LSTMModel, self).__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(hidden_size*2, output_size)

        def forward(self, x):
            # LSTM layer
            lstm_out, _ = self.lstm(x)
            # Use the output of the last LSTM time step
            lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
            # Fully connected layer
            out = self.fc(lstm_out_last)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts):
        """
        Train the LSTM model using random curriculum learning where data is shuffled and split into 25% increments.
        """
        hidden_size = 128
        num_layers = 2
        batch_size = 8
        max_learning_rate = 5e-4
        min_learning_rate = 5e-5
        epochs = epoch_counts

        # Initialize the full dataset
        full_dataset = AudioDataset(audio_folder)
        input_size = full_dataset.input_size
        output_size = len(full_dataset.label_map)
        scaler = full_dataset.scaler
        
        # Shuffle the dataset indices to create random curriculum
        dataset_size = len(full_dataset)
        indices = list(range(dataset_size))
        random.shuffle(indices)
        
        # Split into 25% increments
        split_points = [int(dataset_size * 0.25), int(dataset_size * 0.5), int(dataset_size * 0.75)]
        stage_indices = [
            indices[:split_points[0]],  # First 25% - "Easy"
            indices[:split_points[1]],  # First 50% - "Easy" + "Borderline Easy"
            indices[:split_points[2]],  # First 75% - "Easy" + "Borderline Easy" + "Borderline Tough"
            indices  # 100% - All data ("Tough" added)
        ]
        
        stage_names = [
            "Random Easy (25%)",
            "Random Easy + Borderline Easy (50%)",
            "Random Easy + Borderline Easy + Borderline Tough (75%)",
            "Random Easy + Borderline Easy + Borderline Tough + Tough (100%)"
        ]
        
        print(f"Number of total Training data: {dataset_size}")
        print(f"Data splits: {[len(indices) for indices in stage_indices]}")

        # Initialize the model and optimizer
        model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

        # Training across stages
        for stage, (indices, stage_name) in enumerate(zip(stage_indices, stage_names), 1):
            print(f"\nTraining Stage {stage} with {stage_name} for random curriculum")

            for param_group in optimizer.param_groups:
                param_group['lr'] = max_learning_rate

            # Initialize scheduler for the current stage
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 
                                                                T_max=epoch_counts[stage - 1], 
                                                                eta_min=min_learning_rate)
            
            # Create subset and dataloader for the current stage
            subset = Subset(full_dataset, indices)
            print(f"Number of Training data for stage {stage}: {len(subset)}")

            train_loader = DataLoader(subset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

            # Training loop for current stage
            loss_values = []
            for epoch in range(epochs[stage - 1]):
                model.train()
                running_loss = 0.0
                for i, (inputs, labels) in enumerate(train_loader):  # Ignore difficulty bins in random curriculum
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Forward pass
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimization
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item()

                # Update LR scheduler
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']

                epoch_loss = running_loss / len(train_loader)
                loss_values.append(epoch_loss)
                print(f"Stage {stage}, Epoch {epoch + 1}/{epochs[stage - 1]}, Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}")

            # Save model, loss, and scaler for the current stage
            save_dir = f"saved_weights/subject_independent/{scenario_name}/lstm/random_seed_{seed_value}/stage_{stage}"
            os.makedirs(save_dir, exist_ok=True)
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': scaler
            }
            torch.save(checkpoint, f"{save_dir}/model_checkpoint.pth")
            print(f"Checkpoint saved for random curriculum, Stage {stage}")

    # Define training scenarios
    scenarios = {
        "random_curriculum": [50, 50, 50, 50] 
    }

    # Start training
    audio_folder = 'subject_ordered_features/Training_SI'
    for scenario_name, epoch_counts in scenarios.items():
        print(f"\nStarting training for {scenario_name} with epoch counts: {epoch_counts}")
        random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts)

random_seed = 1737200047

print(f"Starting trial with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Independent - Testing***

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
from torch.utils.data import Dataset
import seaborn as sns

random_seed = 1737200047
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size*2, output_size)

    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        # Use the output of the last LSTM time step
        lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
        # Fully connected layer
        out = self.fc(lstm_out_last)
        return out

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=None, modality="audio"):
        """
        Args:
        - audio_folder: Path to the testing data folder.
        - curriculum_csv_file: CSV file with difficulty scores and bins.
        - scaler: Fitted scaler from the training data.
        - difficulty_score: The difficulty score being evaluated (e.g., "true_label_score").
        - bin_filter: List of bins to filter testing data by (e.g., ["Easy"]).
        - modality: Modality of the testing data (e.g., "audio").
        """
        self.audio_folder = audio_folder
        self.csv_file = curriculum_csv_file
        self.scaler = scaler
        self.difficulty_score = difficulty_score
        self.bin_filter = bin_filter
        self.modality = modality

        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()

        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        # Read and filter CSV data
        df = pd.read_csv(self.csv_file)
        df = df[df['modality'] == self.modality]  # Filter by modality

        # Map difficulty scores to their corresponding bin columns
        bin_column_mapping = {
            'intended_emotion_score': 'bin_intended_emotion_score',
            'entropy_score': 'bin_entropy_score',
            'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
            'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
            'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
        }
        bin_column = bin_column_mapping.get(self.difficulty_score)

        if self.bin_filter:
            df = df[df[bin_column].isin(self.bin_filter)]  # Filter by the selected bins

        file_paths = []
        labels = []

        for _, row in df.iterrows():
            subject_id = row['fileName'].split('_')[0]
            filename = os.path.basename(row['fileName'])
            full_path = os.path.join(self.audio_folder, subject_id, 'Audio', f"{filename}.csv")

            if os.path.exists(full_path):
                full_path = os.path.join('hubert_features', f"{filename}.npy")
                file_paths.append(full_path)
                labels.append(self.extract_label_from_filename(filename))

        return file_paths, labels

    def extract_label_from_filename(self, filename):
        return filename.split('_')[2]

    def load_csv_data(self, file_path):
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)

    def get_counts(self):
        return len(self.file_paths)

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []
    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

def print_mean_macro_accuracies(mean_macro_accuracies):
    print("\n" + "="*80)
    print("MEAN MACRO ACCURACIES ACROSS TRIALS")
    print("="*80)
    
    for difficulty_score, stages_data in mean_macro_accuracies.items():
        print(f"\nDifficulty Score: {difficulty_score}")
        print("-"*60)
        
        # Print overall macro accuracy
        print("\nOverall Performance:")
        for stage, acc_data in stages_data['overall'].items():
            mean_acc = acc_data['mean']
            std_dev = acc_data['std']
            print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")
        
        # Print bin-wise macro accuracy
        bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
        for bin_name in bins:
            print(f"\nBin: {bin_name}")
            for stage, acc_data in stages_data['bins'][bin_name].items():
                mean_acc = acc_data['mean']
                std_dev = acc_data['std']
                print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")

# Main testing logic
def test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name='intended_emotion_lstm'):
    """
    Test the model for all difficulty scores at each stage of training across bins and on the overall test set.
    Args:
    - checkpoint_base_dir: Base directory containing model checkpoints for each difficulty score.
    - test_folder: Path to the folder containing test data.
    - curriculum_csv_file: Path to the CSV file with difficulty scores and bins.
    - device: Device to run the evaluation on (CPU/GPU).
    - difficulty_scores: List of difficulty scores to evaluate.
    """
    
    # Initialize dictionary to store macro accuracies across trials
    all_trials_macro_acc = {
        score: {
            'overall': {stage: [] for stage in range(1, 5)},
            'bins': {
                bin_name: {stage: [] for stage in range(1, 5)} 
                for bin_name in ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
            }
        } for score in difficulty_scores
    }
    
    print(f'\n{"="*60}')
    print(f'Results for Random Seed: {random_seed}')
    print(f'{"="*60}')

    bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
    results = {}

    for difficulty_score in difficulty_scores:
        print(f"\nStarting evaluation for difficulty score: {scenario_name}")
        checkpoint_dir = os.path.join(checkpoint_base_dir, scenario_name)
        results[difficulty_score] = {
            "overall": [],
            "macro_acc": [],
            "bal_acc": [],
            "bins": {bin_name: [] for bin_name in bins},
            "bins_macro": {bin_name: [] for bin_name in bins},
            "counts": {bin_name: 0 for bin_name in bins},
            "total_count": 0
        }

        loss_curves = []

        for stage in range(1, 5):
            print(f"\nEvaluating model after Stage {stage} training for {scenario_name}...")
                
            checkpoint_path = os.path.join(checkpoint_dir, f"lstm/random_seed_{random_seed}/stage_{stage}/model_checkpoint.pth")
            checkpoint = torch.load(checkpoint_path, weights_only=False)

            # Load model
            input_size = checkpoint['model_state_dict']['lstm.weight_ih_l0'].shape[1]
            hidden_size = 128
            num_layers = 2
            output_size = len(checkpoint['model_state_dict']['fc.bias'])

            model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
            model.load_state_dict(checkpoint['model_state_dict'])
                

            # Load scaler from checkpoint
            scaler = checkpoint['scaler']

            # Overall performance
            overall_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score)
            overall_accuracy, macro_acc, bal_acc, overall_report, overall_labels, overall_preds = evaluate_model(model, overall_dataset, device)
            overall_count = overall_dataset.get_counts()
            results[difficulty_score]["overall"].append(overall_accuracy)
            results[difficulty_score]["macro_acc"].append(macro_acc)
            results[difficulty_score]["bal_acc"].append(bal_acc)
            results[difficulty_score]["total_count"] = overall_count
                
            # Store macro accuracy for this trial and stage
            all_trials_macro_acc[difficulty_score]['overall'][stage].append(macro_acc)
                
            print(f"Overall accuracy after Stage {stage}: {overall_accuracy:.4f}")
            print(f"Macro accuracy after Stage {stage}: {macro_acc:.4f}")
            print(f"Balanced accuracy after Stage {stage}: {bal_acc:.4f}")
            print(f"Total test data points: {overall_count}")

            print("Classification Report:")
            print(overall_report)
                
            # Plot confusion matrix for the final stage
            if stage == 4:
                plot_confusion_matrix(overall_labels, overall_preds, 
                                    overall_dataset.label_map,
                                    title=f'Confusion Matrix {scenario_name} (LSTMs)')

            # Bin-wise performance
            for bin_name in bins:
                bin_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=[bin_name])
                bin_accuracy, macro_acc, _, bin_report, bin_labels, bin_preds = evaluate_model(model, bin_dataset, device)
                bin_count = bin_dataset.get_counts()
                results[difficulty_score]["bins"][bin_name].append(bin_accuracy)
                results[difficulty_score]["bins_macro"][bin_name].append(macro_acc)
                results[difficulty_score]["counts"][bin_name] = bin_count
                    
                # Store bin-wise macro accuracy for this trial and stage
                all_trials_macro_acc[difficulty_score]['bins'][bin_name][stage].append(macro_acc)
                    
                print(f"Std. Accuracy for bin {bin_name} after Stage {stage}: {bin_accuracy:.4f}")
                print(f"Macro Accuracy for bin {bin_name} after Stage {stage}: {macro_acc:.4f}")
                print(f"Number of test data points in bin {bin_name}: {bin_count}")

            # Extract loss values for plotting
            loss_values = checkpoint.get('loss_values', None)
            if loss_values:
                loss_curves.append(loss_values)
            else:
                print(f"Loss values not found in checkpoint for Stage {stage}")

        # Plot loss curves for the difficulty score
        plt.figure(figsize=(10, 6))
        for stage, loss_values in enumerate(loss_curves, start=1):
            plt.plot(range(len(loss_values)), loss_values, label=f"Stage {stage}")
        plt.title(f"Training Loss Curve for {scenario_name}")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["overall"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Performance Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot macro accuracy results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["macro_acc"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins_macro"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Macro Accuracy Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Macro Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

# Paths
scenario_name = "random_curriculum"
checkpoint_base_dir = f"saved_weights/subject_independent"
test_folder = "subject_ordered_features/Testing_SI"
curriculum_csv_file = "curriculum_dataset_results_binned.csv"
difficulty_scores = {
    'intended_emotion_score': 'bin_intended_emotion_score',
    # 'entropy_score': 'bin_entropy_score',
    # 'intended_emotion_similarity_score': 'bin_intended_emotion_similarity_score',
    # 'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
    # 'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
    # 'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
}

# Run the test
test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name=scenario_name)

***<H1>Transformer Based</H1>***

***Subject Dependent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_trial(seed_value = 1737200047):

    set_seed(seed_value)
   
    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels and difficulty bins

    # Positional Encoding for Transformer
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000):
            super(PositionalEncoding, self).__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            pe = pe.unsqueeze(0)
            self.register_buffer('pe', pe)

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]

    # Transformer Encoder Model
    class TransformerEncoderModel(nn.Module):
        def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
            super(TransformerEncoderModel, self).__init__()
            self.d_model = d_model
            self.embedding = nn.Linear(input_size, d_model)
            self.pos_encoder = PositionalEncoding(d_model)
            encoder_layers = nn.TransformerEncoderLayer(
                d_model, 
                nhead, 
                dim_feedforward=d_model*4, 
                dropout=dropout,
                batch_first=True  
            )
            self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
            self.fc = nn.Linear(d_model, output_size)
            
        def forward(self, x):
            # x shape: (batch_size, seq_len, input_size)
            x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
            x = self.pos_encoder(x)
            x = self.transformer_encoder(x)
            x = x.mean(dim=1)  # Average over sequence length
            return self.fc(x) 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts, modality="audio"):
        """
        Train the LSTM model using random curriculum learning where data is shuffled and split into 25% increments.
        """
        d_model = 128
        nhead = 4
        num_layers = 2
        batch_size = 8
        max_learning_rate = 5e-4
        min_learning_rate = 5e-5
        epochs = epoch_counts

        # Initialize the full dataset
        full_dataset = AudioDataset(audio_folder)
        input_size = full_dataset.input_size
        output_size = len(full_dataset.label_map)
        scaler = full_dataset.scaler
        
        # Shuffle the dataset indices to create random curriculum
        dataset_size = len(full_dataset)
        indices = list(range(dataset_size))
        random.shuffle(indices)
        
        # Split into 25% increments
        split_points = [int(dataset_size * 0.25), int(dataset_size * 0.5), int(dataset_size * 0.75)]
        stage_indices = [
            indices[:split_points[0]],  # First 25% - "Easy"
            indices[:split_points[1]],  # First 50% - "Easy" + "Borderline Easy"
            indices[:split_points[2]],  # First 75% - "Easy" + "Borderline Easy" + "Borderline Tough"
            indices  # 100% - All data ("Tough" added)
        ]
        
        stage_names = [
            "Random Easy (25%)",
            "Random Easy + Borderline Easy (50%)",
            "Random Easy + Borderline Easy + Borderline Tough (75%)",
            "Random Easy + Borderline Easy + Borderline Tough + Tough (100%)"
        ]
        
        print(f"Number of total Training data: {dataset_size}")
        print(f"Data splits: {[len(indices) for indices in stage_indices]}")

        # Initialize the model and optimizer
        model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

        # Training across stages
        for stage, (indices, stage_name) in enumerate(zip(stage_indices, stage_names), 1):
            print(f"\nTraining Stage {stage} with {stage_name} for random curriculum")

            for param_group in optimizer.param_groups:
                param_group['lr'] = max_learning_rate

            # Initialize scheduler for the current stage
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 
                                                                T_max=epoch_counts[stage - 1], 
                                                                eta_min=min_learning_rate)
            
            # Create subset and dataloader for the current stage
            subset = Subset(full_dataset, indices)
            print(f"Number of Training data for stage {stage}: {len(subset)}")

            train_loader = DataLoader(subset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

            # Training loop for current stage
            loss_values = []
            for epoch in range(epochs[stage - 1]):
                model.train()
                running_loss = 0.0
                for i, (inputs, labels) in enumerate(train_loader):  # Ignore difficulty bins in random curriculum
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Forward pass
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimization
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item()

                # Update LR scheduler
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']

                epoch_loss = running_loss / len(train_loader)
                loss_values.append(epoch_loss)
                print(f"Stage {stage}, Epoch {epoch + 1}/{epochs[stage - 1]}, Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}")

            # Save model, loss, and scaler for the current stage
            save_dir = f"saved_weights/subject_dependent/{scenario_name}/transformer/random_seed_{seed_value}/stage_{stage}"
            os.makedirs(save_dir, exist_ok=True)
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': scaler
            }
            torch.save(checkpoint, f"{save_dir}/model_checkpoint.pth")
            print(f"Checkpoint saved for random curriculum, Stage {stage}")

    # Define training scenarios
    scenarios = {
        "random_curriculum": [100, 100, 100, 100]
    }

    # Start training
    audio_folder = 'subject_ordered_features/Training'
    for scenario_name, epoch_counts in scenarios.items():
        print(f"\nStarting training for {scenario_name} with epoch counts: {epoch_counts}")
        random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts, modality="audio")

random_seed = 1737200047

print(f"Starting trial with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Dependent - Testing***

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
from torch.utils.data import Dataset
import seaborn as sns

random_seed = 1737200047
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Positional Encoding for Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Transformer Encoder Model
class TransformerEncoderModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
        super(TransformerEncoderModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Average over sequence length
        return self.fc(x)

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=None, modality="audio"):
        """
        Args:
        - audio_folder: Path to the testing data folder.
        - curriculum_csv_file: CSV file with difficulty scores and bins.
        - scaler: Fitted scaler from the training data.
        - difficulty_score: The difficulty score being evaluated (e.g., "true_label_score").
        - bin_filter: List of bins to filter testing data by (e.g., ["Easy"]).
        - modality: Modality of the testing data (e.g., "audio").
        """
        self.audio_folder = audio_folder
        self.csv_file = curriculum_csv_file
        self.scaler = scaler
        self.difficulty_score = difficulty_score
        self.bin_filter = bin_filter
        self.modality = modality

        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()

        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        # Read and filter CSV data
        df = pd.read_csv(self.csv_file)
        df = df[df['modality'] == self.modality]  # Filter by modality

        # Map difficulty scores to their corresponding bin columns
        bin_column_mapping = {
            'intended_emotion_score': 'bin_intended_emotion_score',
            'entropy_score': 'bin_entropy_score',
            'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
            'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
            'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
        }
        bin_column = bin_column_mapping.get(self.difficulty_score)

        if self.bin_filter:
            df = df[df[bin_column].isin(self.bin_filter)]  # Filter by the selected bins

        file_paths = []
        labels = []

        for _, row in df.iterrows():
            subject_id = row['fileName'].split('_')[0]
            filename = os.path.basename(row['fileName'])
            full_path = os.path.join(self.audio_folder, subject_id, 'Audio', f"{filename}.csv")

            if os.path.exists(full_path):
                full_path = os.path.join('hubert_features', f"{filename}.npy")
                file_paths.append(full_path)
                labels.append(self.extract_label_from_filename(filename))

        return file_paths, labels

    def extract_label_from_filename(self, filename):
        return filename.split('_')[2]

    def load_csv_data(self, file_path):
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)

    def get_counts(self):
        return len(self.file_paths)

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []
    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

def print_mean_macro_accuracies(mean_macro_accuracies):
    print("\n" + "="*80)
    print("MEAN MACRO ACCURACIES ACROSS TRIALS")
    print("="*80)
    
    for difficulty_score, stages_data in mean_macro_accuracies.items():
        print(f"\nDifficulty Score: {difficulty_score}")
        print("-"*60)
        
        # Print overall macro accuracy
        print("\nOverall Performance:")
        for stage, acc_data in stages_data['overall'].items():
            mean_acc = acc_data['mean']
            std_dev = acc_data['std']
            print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")
        
        # Print bin-wise macro accuracy
        bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
        for bin_name in bins:
            print(f"\nBin: {bin_name}")
            for stage, acc_data in stages_data['bins'][bin_name].items():
                mean_acc = acc_data['mean']
                std_dev = acc_data['std']
                print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")

# Main testing logic
def test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name='intended_emotion_lstm'):
    """
    Test the model for all difficulty scores at each stage of training across bins and on the overall test set.
    Args:
    - checkpoint_base_dir: Base directory containing model checkpoints for each difficulty score.
    - test_folder: Path to the folder containing test data.
    - curriculum_csv_file: Path to the CSV file with difficulty scores and bins.
    - device: Device to run the evaluation on (CPU/GPU).
    - difficulty_scores: List of difficulty scores to evaluate.
    """
    
    # Initialize dictionary to store macro accuracies across trials
    all_trials_macro_acc = {
        score: {
            'overall': {stage: [] for stage in range(1, 5)},
            'bins': {
                bin_name: {stage: [] for stage in range(1, 5)} 
                for bin_name in ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
            }
        } for score in difficulty_scores
    }
    
    print(f'\n{"="*60}')
    print(f'Results for Random Seed: {random_seed}')
    print(f'{"="*60}')

    bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
    results = {}

    for difficulty_score in difficulty_scores:
        print(f"\nStarting evaluation for difficulty score: {scenario_name}")
        checkpoint_dir = os.path.join(checkpoint_base_dir, scenario_name)
        results[difficulty_score] = {
            "overall": [],
            "macro_acc": [],
            "bal_acc": [],
            "bins": {bin_name: [] for bin_name in bins},
            "bins_macro": {bin_name: [] for bin_name in bins},
            "counts": {bin_name: 0 for bin_name in bins},
            "total_count": 0
        }

        loss_curves = []

        for stage in range(1, 5):
            print(f"\nEvaluating model after Stage {stage} training for {scenario_name}...")
                
            checkpoint_path = os.path.join(checkpoint_dir, f"transformer/random_seed_{random_seed}/stage_{stage}/model_checkpoint.pth")
            checkpoint = torch.load(checkpoint_path, weights_only=False)

            # Get model parameters from the checkpoint
            input_size = checkpoint['model_state_dict']['embedding.weight'].shape[1]  # Get input size from model weights
            d_model = 128  # Match with your training parameters
            nhead = 4
            num_layers = 2
            output_size = 6

            # Initialize the model and load the state dict
            model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
            model.load_state_dict(checkpoint['model_state_dict'])
                

            # Load scaler from checkpoint
            scaler = checkpoint['scaler']

            # Overall performance
            overall_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score)
            overall_accuracy, macro_acc, bal_acc, overall_report, overall_labels, overall_preds = evaluate_model(model, overall_dataset, device)
            overall_count = overall_dataset.get_counts()
            results[difficulty_score]["overall"].append(overall_accuracy)
            results[difficulty_score]["macro_acc"].append(macro_acc)
            results[difficulty_score]["bal_acc"].append(bal_acc)
            results[difficulty_score]["total_count"] = overall_count
                
            # Store macro accuracy for this trial and stage
            all_trials_macro_acc[difficulty_score]['overall'][stage].append(macro_acc)
                
            print(f"Overall accuracy after Stage {stage}: {overall_accuracy:.4f}")
            print(f"Macro accuracy after Stage {stage}: {macro_acc:.4f}")
            print(f"Balanced accuracy after Stage {stage}: {bal_acc:.4f}")
            print(f"Total test data points: {overall_count}")

            print("Classification Report:")
            print(overall_report)
                
            # Plot confusion matrix for the final stage
            if stage == 4:
                plot_confusion_matrix(overall_labels, overall_preds, 
                                    overall_dataset.label_map,
                                    title=f'Confusion Matrix {scenario_name} (Transformer)')

            # Bin-wise performance
            for bin_name in bins:
                bin_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=[bin_name])
                bin_accuracy, macro_acc, _, bin_report, bin_labels, bin_preds = evaluate_model(model, bin_dataset, device)
                bin_count = bin_dataset.get_counts()
                results[difficulty_score]["bins"][bin_name].append(bin_accuracy)
                results[difficulty_score]["bins_macro"][bin_name].append(macro_acc)
                results[difficulty_score]["counts"][bin_name] = bin_count
                    
                # Store bin-wise macro accuracy for this trial and stage
                all_trials_macro_acc[difficulty_score]['bins'][bin_name][stage].append(macro_acc)
                    
                print(f"Std. Accuracy for bin {bin_name} after Stage {stage}: {bin_accuracy:.4f}")
                print(f"Macro Accuracy for bin {bin_name} after Stage {stage}: {macro_acc:.4f}")
                print(f"Number of test data points in bin {bin_name}: {bin_count}")

            # Extract loss values for plotting
            loss_values = checkpoint.get('loss_values', None)
            if loss_values:
                loss_curves.append(loss_values)
            else:
                print(f"Loss values not found in checkpoint for Stage {stage}")

        # Plot loss curves for the difficulty score
        plt.figure(figsize=(10, 6))
        for stage, loss_values in enumerate(loss_curves, start=1):
            plt.plot(range(len(loss_values)), loss_values, label=f"Stage {stage}")
        plt.title(f"Training Loss Curve for {scenario_name}")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["overall"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Performance Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot macro accuracy results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["macro_acc"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins_macro"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Macro Accuracy Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Macro Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

# Paths
scenario_name = "random_curriculum"
checkpoint_base_dir = f"saved_weights/subject_dependent"
test_folder = "subject_ordered_features/Testing"
curriculum_csv_file = "curriculum_dataset_results_binned.csv"
difficulty_scores = {
    'intended_emotion_score': 'bin_intended_emotion_score',
    # 'entropy_score': 'bin_entropy_score',
    # 'intended_emotion_similarity_score': 'bin_intended_emotion_similarity_score',
    # 'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
    # 'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
    # 'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
}

# Run the test
test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name=scenario_name)

***Subject Independent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_trial(seed_value = 1737200047):

    set_seed(seed_value)
   
    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels and difficulty bins

    # Positional Encoding for Transformer
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000):
            super(PositionalEncoding, self).__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            pe = pe.unsqueeze(0)
            self.register_buffer('pe', pe)

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]

    # Transformer Encoder Model
    class TransformerEncoderModel(nn.Module):
        def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
            super(TransformerEncoderModel, self).__init__()
            self.d_model = d_model
            self.embedding = nn.Linear(input_size, d_model)
            self.pos_encoder = PositionalEncoding(d_model)
            encoder_layers = nn.TransformerEncoderLayer(
                d_model, 
                nhead, 
                dim_feedforward=d_model*4, 
                dropout=dropout,
                batch_first=True  
            )
            self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
            self.fc = nn.Linear(d_model, output_size)
            
        def forward(self, x):
            # x shape: (batch_size, seq_len, input_size)
            x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
            x = self.pos_encoder(x)
            x = self.transformer_encoder(x)
            x = x.mean(dim=1)  # Average over sequence length
            return self.fc(x) 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts, modality="audio"):
        """
        Train the LSTM model using random curriculum learning where data is shuffled and split into 25% increments.
        """
        d_model = 128
        nhead = 4
        num_layers = 2
        batch_size = 8
        max_learning_rate = 5e-4
        min_learning_rate = 5e-5
        epochs = epoch_counts

        # Initialize the full dataset
        full_dataset = AudioDataset(audio_folder)
        input_size = full_dataset.input_size
        output_size = len(full_dataset.label_map)
        scaler = full_dataset.scaler
        
        # Shuffle the dataset indices to create random curriculum
        dataset_size = len(full_dataset)
        indices = list(range(dataset_size))
        random.shuffle(indices)
        
        # Split into 25% increments
        split_points = [int(dataset_size * 0.25), int(dataset_size * 0.5), int(dataset_size * 0.75)]
        stage_indices = [
            indices[:split_points[0]],  # First 25% - "Easy"
            indices[:split_points[1]],  # First 50% - "Easy" + "Borderline Easy"
            indices[:split_points[2]],  # First 75% - "Easy" + "Borderline Easy" + "Borderline Tough"
            indices  # 100% - All data ("Tough" added)
        ]
        
        stage_names = [
            "Random Easy (25%)",
            "Random Easy + Borderline Easy (50%)",
            "Random Easy + Borderline Easy + Borderline Tough (75%)",
            "Random Easy + Borderline Easy + Borderline Tough + Tough (100%)"
        ]
        
        print(f"Number of total Training data: {dataset_size}")
        print(f"Data splits: {[len(indices) for indices in stage_indices]}")

        # Initialize the model and optimizer
        model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

        # Training across stages
        for stage, (indices, stage_name) in enumerate(zip(stage_indices, stage_names), 1):
            print(f"\nTraining Stage {stage} with {stage_name} for random curriculum")

            for param_group in optimizer.param_groups:
                param_group['lr'] = max_learning_rate

            # Initialize scheduler for the current stage
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 
                                                                T_max=epoch_counts[stage - 1], 
                                                                eta_min=min_learning_rate)
            
            # Create subset and dataloader for the current stage
            subset = Subset(full_dataset, indices)
            print(f"Number of Training data for stage {stage}: {len(subset)}")

            train_loader = DataLoader(subset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

            # Training loop for current stage
            loss_values = []
            for epoch in range(epochs[stage - 1]):
                model.train()
                running_loss = 0.0
                for i, (inputs, labels) in enumerate(train_loader):  # Ignore difficulty bins in random curriculum
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Forward pass
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimization
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item()

                # Update LR scheduler
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']

                epoch_loss = running_loss / len(train_loader)
                loss_values.append(epoch_loss)
                print(f"Stage {stage}, Epoch {epoch + 1}/{epochs[stage - 1]}, Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}")

            # Save model, loss, and scaler for the current stage
            save_dir = f"saved_weights/subject_independent/{scenario_name}/transformer/random_seed_{seed_value}/stage_{stage}"
            os.makedirs(save_dir, exist_ok=True)
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': scaler
            }
            torch.save(checkpoint, f"{save_dir}/model_checkpoint.pth")
            print(f"Checkpoint saved for random curriculum, Stage {stage}")

    # Define training scenarios
    scenarios = {
        "random_curriculum": [100, 100, 100, 100]
    }

    # Start training
    audio_folder = 'subject_ordered_features/Training_SI'
    for scenario_name, epoch_counts in scenarios.items():
        print(f"\nStarting training for {scenario_name} with epoch counts: {epoch_counts}")
        random_curriculum_learning_training(audio_folder, scenario_name, epoch_counts, modality="audio")

random_seed = 1737200047

print(f"Starting trial with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Independent - Testing***

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
from torch.utils.data import Dataset
import seaborn as sns

random_seed = 1737200047
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Positional Encoding for Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Transformer Encoder Model
class TransformerEncoderModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
        super(TransformerEncoderModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Average over sequence length
        return self.fc(x)

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=None, modality="audio"):
        """
        Args:
        - audio_folder: Path to the testing data folder.
        - curriculum_csv_file: CSV file with difficulty scores and bins.
        - scaler: Fitted scaler from the training data.
        - difficulty_score: The difficulty score being evaluated (e.g., "true_label_score").
        - bin_filter: List of bins to filter testing data by (e.g., ["Easy"]).
        - modality: Modality of the testing data (e.g., "audio").
        """
        self.audio_folder = audio_folder
        self.csv_file = curriculum_csv_file
        self.scaler = scaler
        self.difficulty_score = difficulty_score
        self.bin_filter = bin_filter
        self.modality = modality

        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()

        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        # Read and filter CSV data
        df = pd.read_csv(self.csv_file)
        df = df[df['modality'] == self.modality]  # Filter by modality

        # Map difficulty scores to their corresponding bin columns
        bin_column_mapping = {
            'intended_emotion_score': 'bin_intended_emotion_score',
            'entropy_score': 'bin_entropy_score',
            'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
            'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
            'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
        }
        bin_column = bin_column_mapping.get(self.difficulty_score)

        if self.bin_filter:
            df = df[df[bin_column].isin(self.bin_filter)]  # Filter by the selected bins

        file_paths = []
        labels = []

        for _, row in df.iterrows():
            subject_id = row['fileName'].split('_')[0]
            filename = os.path.basename(row['fileName'])
            full_path = os.path.join(self.audio_folder, subject_id, 'Audio', f"{filename}.csv")

            if os.path.exists(full_path):
                full_path = os.path.join('hubert_features', f"{filename}.npy")
                file_paths.append(full_path)
                labels.append(self.extract_label_from_filename(filename))

        return file_paths, labels

    def extract_label_from_filename(self, filename):
        return filename.split('_')[2]

    def load_csv_data(self, file_path):
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)

    def get_counts(self):
        return len(self.file_paths)

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []
    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

def print_mean_macro_accuracies(mean_macro_accuracies):
    print("\n" + "="*80)
    print("MEAN MACRO ACCURACIES ACROSS TRIALS")
    print("="*80)
    
    for difficulty_score, stages_data in mean_macro_accuracies.items():
        print(f"\nDifficulty Score: {difficulty_score}")
        print("-"*60)
        
        # Print overall macro accuracy
        print("\nOverall Performance:")
        for stage, acc_data in stages_data['overall'].items():
            mean_acc = acc_data['mean']
            std_dev = acc_data['std']
            print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")
        
        # Print bin-wise macro accuracy
        bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
        for bin_name in bins:
            print(f"\nBin: {bin_name}")
            for stage, acc_data in stages_data['bins'][bin_name].items():
                mean_acc = acc_data['mean']
                std_dev = acc_data['std']
                print(f"Stage {stage}: Mean = {mean_acc:.4f}, Std = {std_dev:.4f}")

# Main testing logic
def test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name='intended_emotion_lstm'):
    """
    Test the model for all difficulty scores at each stage of training across bins and on the overall test set.
    Args:
    - checkpoint_base_dir: Base directory containing model checkpoints for each difficulty score.
    - test_folder: Path to the folder containing test data.
    - curriculum_csv_file: Path to the CSV file with difficulty scores and bins.
    - device: Device to run the evaluation on (CPU/GPU).
    - difficulty_scores: List of difficulty scores to evaluate.
    """
    
    # Initialize dictionary to store macro accuracies across trials
    all_trials_macro_acc = {
        score: {
            'overall': {stage: [] for stage in range(1, 5)},
            'bins': {
                bin_name: {stage: [] for stage in range(1, 5)} 
                for bin_name in ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
            }
        } for score in difficulty_scores
    }
    
    print(f'\n{"="*60}')
    print(f'Results for Random Seed: {random_seed}')
    print(f'{"="*60}')

    bins = ["Easy", "Borderline Easy", "Borderline Tough", "Tough"]
    results = {}

    for difficulty_score in difficulty_scores:
        print(f"\nStarting evaluation for difficulty score: {scenario_name}")
        checkpoint_dir = os.path.join(checkpoint_base_dir, scenario_name)
        results[difficulty_score] = {
            "overall": [],
            "macro_acc": [],
            "bal_acc": [],
            "bins": {bin_name: [] for bin_name in bins},
            "bins_macro": {bin_name: [] for bin_name in bins},
            "counts": {bin_name: 0 for bin_name in bins},
            "total_count": 0
        }

        loss_curves = []

        for stage in range(1, 5):
            print(f"\nEvaluating model after Stage {stage} training for {scenario_name}...")
                
            checkpoint_path = os.path.join(checkpoint_dir, f"transformer/random_seed_{random_seed}/stage_{stage}/model_checkpoint.pth")
            checkpoint = torch.load(checkpoint_path, weights_only=False)

            # Get model parameters from the checkpoint
            input_size = checkpoint['model_state_dict']['embedding.weight'].shape[1]  # Get input size from model weights
            d_model = 128  # Match with your training parameters
            nhead = 4
            num_layers = 2
            output_size = 6

            # Initialize the model and load the state dict
            model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
            model.load_state_dict(checkpoint['model_state_dict'])
                

            # Load scaler from checkpoint
            scaler = checkpoint['scaler']

            # Overall performance
            overall_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score)
            overall_accuracy, macro_acc, bal_acc, overall_report, overall_labels, overall_preds = evaluate_model(model, overall_dataset, device)
            overall_count = overall_dataset.get_counts()
            results[difficulty_score]["overall"].append(overall_accuracy)
            results[difficulty_score]["macro_acc"].append(macro_acc)
            results[difficulty_score]["bal_acc"].append(bal_acc)
            results[difficulty_score]["total_count"] = overall_count
                
            # Store macro accuracy for this trial and stage
            all_trials_macro_acc[difficulty_score]['overall'][stage].append(macro_acc)
                
            print(f"Overall accuracy after Stage {stage}: {overall_accuracy:.4f}")
            print(f"Macro accuracy after Stage {stage}: {macro_acc:.4f}")
            print(f"Balanced accuracy after Stage {stage}: {bal_acc:.4f}")
            print(f"Total test data points: {overall_count}")

            print("Classification Report:")
            print(overall_report)
                
            # Plot confusion matrix for the final stage
            if stage == 4:
                plot_confusion_matrix(overall_labels, overall_preds, 
                                    overall_dataset.label_map,
                                    title=f'Confusion Matrix {scenario_name} (Transformer)')

            # Bin-wise performance
            for bin_name in bins:
                bin_dataset = TestDataset(test_folder, curriculum_csv_file, scaler, difficulty_score, bin_filter=[bin_name])
                bin_accuracy, macro_acc, _, bin_report, bin_labels, bin_preds = evaluate_model(model, bin_dataset, device)
                bin_count = bin_dataset.get_counts()
                results[difficulty_score]["bins"][bin_name].append(bin_accuracy)
                results[difficulty_score]["bins_macro"][bin_name].append(macro_acc)
                results[difficulty_score]["counts"][bin_name] = bin_count
                    
                # Store bin-wise macro accuracy for this trial and stage
                all_trials_macro_acc[difficulty_score]['bins'][bin_name][stage].append(macro_acc)
                    
                print(f"Std. Accuracy for bin {bin_name} after Stage {stage}: {bin_accuracy:.4f}")
                print(f"Macro Accuracy for bin {bin_name} after Stage {stage}: {macro_acc:.4f}")
                print(f"Number of test data points in bin {bin_name}: {bin_count}")

            # Extract loss values for plotting
            loss_values = checkpoint.get('loss_values', None)
            if loss_values:
                loss_curves.append(loss_values)
            else:
                print(f"Loss values not found in checkpoint for Stage {stage}")

        # Plot loss curves for the difficulty score
        plt.figure(figsize=(10, 6))
        for stage, loss_values in enumerate(loss_curves, start=1):
            plt.plot(range(len(loss_values)), loss_values, label=f"Stage {stage}")
        plt.title(f"Training Loss Curve for {scenario_name}")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["overall"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Performance Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

    # Plot macro accuracy results for this trial
    for difficulty_score, result in results.items():
        plt.figure(figsize=(12, 8))
        plt.plot(range(1, 5), result["macro_acc"], label="Overall Performance", marker='o')
        for bin_name in bins:
            plt.plot(range(1, 5), result["bins_macro"][bin_name], label=f"Performance on {bin_name}", marker='o')

        plt.title(f"Macro Accuracy Across Training Stages for {scenario_name}")
        plt.xlabel("Training Stage")
        plt.ylabel("Macro Accuracy")
        plt.legend()
        plt.grid()
        plt.show()

# Paths
scenario_name = "random_curriculum"
checkpoint_base_dir = f"saved_weights/subject_independent"
test_folder = "subject_ordered_features/Testing_SI"
curriculum_csv_file = "curriculum_dataset_results_binned.csv"
difficulty_scores = {
    'intended_emotion_score': 'bin_intended_emotion_score',
    # 'entropy_score': 'bin_entropy_score',
    # 'intended_emotion_similarity_score': 'bin_intended_emotion_similarity_score',
    # 'intended_perceived_agreement_1': 'bin_intended_perceived_agreement_1',
    # 'intended_perceived_agreement_2': 'bin_intended_perceived_agreement_2',
    # 'intended_perceived_agreement_3': 'bin_intended_perceived_agreement_3',  
}

# Run the test
test_curriculum_models(checkpoint_base_dir, test_folder, curriculum_csv_file, device, difficulty_scores, scenario_name=scenario_name)